<a href="https://colab.research.google.com/github/Rajukc45/Capstone-project/blob/main/week%207%20%268%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [1]:
!pip install google-generativeai scikit-learn joblib

In [16]:
import google.generativeai as genai

genai.configure(api_key="PASTE_YOUR_REAL_API_KEY_HERE")

In [18]:
sample = X_test.iloc[[0]]   # keeps column names

In [19]:
def predict_transaction(data):

    data_scaled = scaler.transform(data)

    pred = model.predict(data_scaled)[0]
    prob = model.predict_proba(data_scaled)[0][1]

    label = "Fraudulent" if pred == 1 else "Legitimate"

    return label, prob

In [3]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])
df.head()

Saving creditcard.csv to creditcard.csv


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


prepare data


In [5]:
# Change 'Class' to your target column if different
X = df.drop("Class", axis=1)
y = df["Class"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale data

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Train Model (Random Forest)

In [7]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

Evaluate Model

In [8]:
y_pred = model.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3262
           1       0.86      0.80      0.83        15

    accuracy                           1.00      3277
   macro avg       0.93      0.90      0.91      3277
weighted avg       1.00      1.00      1.00      3277



Save Model


In [9]:
joblib.dump(model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

Setup Gemini API

In [10]:
genai.configure(api_key="YOUR_GEMINI_API_KEY")

llm = genai.GenerativeModel("gemini-1.5-flash")

Prediction Function

In [22]:
def predict_transaction(data):

    data_scaled = scaler.transform(data)

    pred = model.predict(data_scaled)[0]
    prob = model.predict_proba(data_scaled)[0][1]

    label = "Fraudulent" if pred == 1 else "Legitimate"

    return label, prob

Explanation Function (Gemini)

In [13]:
def explain_transaction(data, prediction, probability):

    prompt = f"""
    You are a fraud detection assistant.

    Transaction data: {data}
    Prediction: {prediction}
    Fraud probability: {probability:.2f}

    Explain in simple terms why this transaction is classified this way.
    Mention possible reasons like unusual amount, time, or behavior.
    Keep it short.
    """

    response = llm.generate_content(prompt)

    return response.text

Full System

In [14]:
def fraud_system(data):
    pred, prob = predict_transaction(data)
    explanation = explain_transaction(data, pred, prob)

    return {
        "Prediction": pred,
        "Probability": prob,
        "Explanation": explanation
    }

Test Example

In [23]:
sample = X_test.iloc[[0]]

result = fraud_system(sample)

print(result)

BadRequest: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.